In [1]:
from dotenv import load_dotenv
import os
import requests
from requests.auth import HTTPBasicAuth
import openmeteo_requests
import requests_cache
from retry_requests import retry
import pandas as pd

In [14]:
load_dotenv(override=True)

CITY = os.getenv("CITY")
USERNAME = os.getenv("USERNAME")
PASSWORD = os.getenv("PASSWORD")
BASE_URL = f"https://{CITY}.pulse.eco/rest"

print(f"City: {CITY}")

City: bitola


In [19]:
def get_sensors():
    url = f"{BASE_URL}/sensor"
    r = requests.get(url, auth=HTTPBasicAuth(USERNAME, PASSWORD))
    if r.status_code != 200:
        raise Exception("Failed to fetch sensors:", r.text)
    return r.json()

valid_statuses = {
    "ACTIVE",
    "ACTIVE_UNCONFIRMED",
    "NOT_CLAIMED",
    "NOT_CLAIMED_UNCONFIRMED"
}


sensors = get_sensors()
filtered = [s for s in sensors if s["status"] in valid_statuses]

print("Total sensors:", len(sensors))
print("Filtered sensors:", len(filtered))
print("Example sensor:", filtered[0])

Total sensors: 22
Filtered sensors: 22
Example sensor: {'sensorId': 'd241a044-0a06-40c2-9d90-c91fd0a95060', 'position': '40.99851020119874,21.243798033728062', 'comments': 'Postaveno od RC Bitola na 19.08.2024', 'type': '3', 'description': 'RC Bitola Nize Pole', 'status': 'ACTIVE'}


In [20]:
sensor_locations = []

for s in filtered:
    lat, lon = map(float, s["position"].split(","))
    
    sensor_locations.append({
        "sensorId": s["sensorId"],
        "lat": lat,
        "lon": lon
    })

print("Sensors with coordinates:", len(sensor_locations))
sensor_locations[:3]

Sensors with coordinates: 22


[{'sensorId': 'd241a044-0a06-40c2-9d90-c91fd0a95060',
  'lat': 40.99851020119874,
  'lon': 21.243798033728062},
 {'sensorId': 'fec52a19-9148-4350-a1b4-ae0da05ee199',
  'lat': 41.02464876476277,
  'lon': 21.320767039678557},
 {'sensorId': 'be427cee-4c3a-4aa2-a1ce-9795a74533be',
  'lat': 41.034465595488754,
  'lon': 21.337752050450533}]

In [26]:
cache_session = requests_cache.CachedSession(".cache", expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)

openmeteo = openmeteo_requests.Client(session=retry_session)

url = "https://archive-api.open-meteo.com/v1/archive"

all_weather = []

for sensor in sensor_locations:
    
    params = {
        "latitude": sensor["lat"],
        "longitude": sensor["lon"],
        "start_date": "2025-12-01",
        "end_date": "2026-03-01",
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "wind_speed_10m",
            "wind_direction_10m",
            "surface_pressure"
        ],
        "timezone": "Europe/Skopje"
    }

    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]

    hourly = response.Hourly()

    timestamps = pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    )

    df = pd.DataFrame({
        "timestamp": timestamps,
        "sensorId": sensor["sensorId"],
        "lat": sensor["lat"],
        "lon": sensor["lon"],
        "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
        "relative_humidity_2m": hourly.Variables(1).ValuesAsNumpy(),
        "wind_speed_10m": hourly.Variables(2).ValuesAsNumpy(),
        "wind_direction_10m": hourly.Variables(3).ValuesAsNumpy(),
        "surface_pressure": hourly.Variables(4).ValuesAsNumpy(),
    })

    all_weather.append(df)

    print("Fetched weather for", sensor["sensorId"])

Fetched weather for d241a044-0a06-40c2-9d90-c91fd0a95060
Fetched weather for fec52a19-9148-4350-a1b4-ae0da05ee199
Fetched weather for be427cee-4c3a-4aa2-a1ce-9795a74533be
Fetched weather for c3f3da9b-9fd3-4037-94d3-598d655e6be9
Fetched weather for d7060523-b163-4cc3-bc94-970dac72a38a
Fetched weather for 87f82783-853b-417d-8964-b5cf11e44873
Fetched weather for e20e9778-a020-4b86-932a-b7ab6a713a00
Fetched weather for 2819ecbb-5de3-4092-aa1d-4ba3a8c40add
Fetched weather for a9a2083f-f086-4fae-bdae-355b391f436b
Fetched weather for a17013e7-8d1d-4b0d-8e2f-e0881dbca3ac
Fetched weather for d851c0b9-990e-41db-9c53-529f88524cf9
Fetched weather for 30dab8a6-ff63-43ce-9a3b-99f1f3f7054d
Fetched weather for 16836a55-7140-43e2-9a63-56fac5cba714
Fetched weather for 2001
Fetched weather for 2002
Fetched weather for 24039f11-a4fc-4b2d-8bc0-6fd36059f117
Fetched weather for 874ff9c6-786d-45fc-a90e-48c7ffe03417
Fetched weather for 23b735ef-a996-4a7f-9998-2aa7e78827b0
Fetched weather for ece1058a-ecab-4736

In [27]:
weather_df = pd.concat(all_weather, ignore_index=True)

print(weather_df.shape)
weather_df.head()

(48048, 9)


,timestamp,sensorId,lat,lon,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure
0,2025-11-30 23:00:00+00:00,d241a044-0a06-40c2-9d90-c91fd0a95060,40.99851,21.243798,0.807,98.924561,8.367580,251.175201,885.730225
1,2025-12-01 00:00:00+00:00,d241a044-0a06-40c2-9d90-c91fd0a95060,40.99851,21.243798,1.057,98.926758,9.299225,255.425735,886.101929
2,2025-12-01 01:00:00+00:00,d241a044-0a06-40c2-9d90-c91fd0a95060,40.99851,21.243798,1.407,98.929802,8.891344,248.629303,886.517639
3,2025-12-01 02:00:00+00:00,d241a044-0a06-40c2-9d90-c91fd0a95060,40.99851,21.243798,1.207,100.000000,8.383054,255.068527,886.516724
4,2025-12-01 03:00:00+00:00,d241a044-0a06-40c2-9d90-c91fd0a95060,40.99851,21.243798,0.907,100.000000,8.209263,254.744827,886.732239


In [28]:
weather_df.to_csv("../data/raw/bitola_sensor_weather_features_online.csv", index=False)